In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
import os
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import matplotlib.pyplot as plt
from sklearn.model_selection import RandomizedSearchCV
import seaborn as sns
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score,confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.model_selection import GridSearchCV, StratifiedShuffleSplit
from xgboost.sklearn import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from scipy.special import entr
from sklearn import preprocessing
import matplotlib.pyplot as plt
import math
import seaborn as sns
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score,confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from IPython.display import display
#import itertools
#from itables import init_notebook_mode
import random
#init_notebook_mode(all_interactive=True)

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.impute import SimpleImputer
from sklearn.manifold import Isomap


In [2]:
print("Hel4d")

Hel4d


In [2]:
import pickle
with open('combined_features_2.pkl', 'rb') as f:
    total = pickle.load(f)

In [3]:
import pandas as pd

df = pd.concat([pd.DataFrame(v) for v in total.values()], keys=total.keys())

df.to_csv('my_dict.csv', index=False)


In [4]:
# Clean up df and total to save memory
del df
del total

In [5]:
# Read my_dict.csv
df = pd.read_csv('my_dict.csv')


In [11]:
# Print different values for label column
user_labels = df['label'].unique()
data = df
# Rename label column to user
#data = data.rename(columns={'label': 'user'})
from sklearn.impute import SimpleImputer
# Preprocess your data: replace missing values and infinite values with the mean of the column
imputer = SimpleImputer(missing_values=np.nan, strategy='mean')
data.iloc[:, :] = imputer.fit_transform(data.replace([np.inf, -np.inf], np.nan))

In [12]:
# Initialize lists to accumulate true labels and predictions
all_y_test = []
all_y_pred = []

# Iterate over all user labels
for target_user in user_labels:
    # Filter data for the target user
    user_data = data[data['user'] == target_user]
    impostor_data = data[data['user'] != target_user]

    # Get the minimum number of samples for balancing
    min_samples = min(user_data.shape[0], impostor_data.shape[0])

    # Sample the data to balance the classes
    balanced_user_data = user_data.sample(min_samples)
    balanced_impostor_data = impostor_data.sample(min_samples)

    # Combine the balanced data and shuffle the rows
    balanced_data = pd.concat([balanced_user_data, balanced_impostor_data]).sample(frac=1).reset_index(drop=True)

    # Replace the user labels
    balanced_data['label'] = np.where(balanced_data['user'] == target_user, 0, 1)

    # Split the data into features and labels
    X = balanced_data.drop(['user', 'label'], axis=1)
    y = balanced_data['label']

    # Split your dataset into training and testing subsets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Train the Random Forest model
    rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_classifier.fit(X_train, y_train)

    # Test the model
    y_pred = rf_classifier.predict(X_test)

    # Evaluate the model for the current user
    print(f"User {target_user}:")
    print(classification_report(y_test, y_pred))

    # Accumulate the true labels and predictions
    all_y_test.extend(y_test)
    all_y_pred.extend(y_pred)

# Print the total classification report
print("Total classification report:")
print(classification_report(all_y_test, all_y_pred))

KeyError: 'user'

In [7]:
from sklearn.manifold import Isomap

# Apply Isomap for dimensionality reduction
isomap = Isomap(n_components=10)
data_transformed = isomap.fit_transform(data.drop([ 'label'], axis=1))
data_transformed = pd.DataFrame(data_transformed)
data_transformed['user'] = data['user']
data_transformed['label'] = data['label']

# Iterate over all user labels
for target_user in user_labels:
    # Filter data for the target user
    user_data = data_transformed[data_transformed['user'] == target_user]
    impostor_data = data_transformed[data_transformed['user'] != target_user]

    # Get the minimum number of samples for balancing
    min_samples = min(user_data.shape[0], impostor_data.shape[0])

    # Sample the data to balance the classes
    balanced_user_data = user_data.sample(min_samples)
    balanced_impostor_data = impostor_data.sample(min_samples)

    # Combine the balanced data and shuffle the rows
    balanced_data = pd.concat([balanced_user_data, balanced_impostor_data]).sample(frac=1).reset_index(drop=True)

    # Replace the user labels
    balanced_data['label'] = np.where(balanced_data['user'] == target_user, 0, 1)

    # Split the data into features and labels
    X = balanced_data.drop(['user', 'label'], axis=1)
    y = balanced_data['label']

    # Split your dataset into training and testing subsets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Train the Random Forest model
    rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_classifier.fit(X_train, y_train)

    # Test the model
    y_pred = rf_classifier.predict(X_test)

    # Evaluate the model for the current user
    print(f"User {target_user}:")
    print(classification_report(y_test, y_pred))

    # Accumulate the true labels and predictions
    all_y_test.extend(y_test)
    all_y_pred.extend(y_pred)

# Print the total classification report
print("Total classification report:")
print(classification_report(all_y_test, all_y_pred))

MemoryError: Unable to allocate 10.9 GiB for an array with shape (38220, 38220) and data type float64

In [26]:
# def run_random_forest(data, user_labels, isomap_applied=False):
#     all_y_test = []
#     all_y_pred = []

#     for target_user in user_labels:
#         user_data = data[data['label'] == target_user]
#         impostor_data = data[data['label'] != target_user]

#         min_samples = min(user_data.shape[0], impostor_data.shape[0])

#         balanced_user_data = user_data.sample(min_samples)
#         balanced_impostor_data = impostor_data.sample(min_samples)

#         balanced_data = pd.concat([balanced_user_data, balanced_impostor_data]).sample(frac=1).reset_index(drop=True)

#         balanced_data['label'] = np.where(balanced_data['label'] == target_user, 0, 1)

#         X = balanced_data.drop(['label'], axis=1)
#         y = balanced_data['label']

#         X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#         rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
#         rf_classifier.fit(X_train, y_train)

#         y_pred = rf_classifier.predict(X_test)

#         all_y_test.extend(y_test)
#         all_y_pred.extend(y_pred)

#     print(f"{'With' if isomap_applied else 'Without'} Isomap:")
#     print(classification_report(all_y_test, all_y_pred))
#     print()


# from sklearn.metrics import roc_curve, auc
# from scipy.optimize import brentq
# from scipy.interpolate import interp1d

# def calculate_eer(y_test, y_score):
#     fpr, tpr, thresholds = roc_curve(y_test, y_score)
#     fnr = 1 - tpr
#     eer = brentq(lambda x : 1. - x - interp1d(fpr, fnr)(x), 0., 1.)
#     return eer

# def run_random_forest(data, user_labels, isomap_applied=False):
#     eer_list = []

#     for target_user in user_labels:
#         user_data = data[data['label'] == target_user]
#         impostor_data = data[data['label'] != target_user]

#         unbalanced_data = pd.concat([user_data, impostor_data]).sample(frac=1).reset_index(drop=True)

#         unbalanced_data['label'] = np.where(unbalanced_data['label'] == target_user, 0, 1)

#         X = unbalanced_data.drop(['label'], axis=1)
#         y = unbalanced_data['label']

#         X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#         rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
#         rf_classifier.fit(X_train, y_train)

#         y_score = rf_classifier.predict_proba(X_test)[:, 1]

#         eer = calculate_eer(y_test, y_score)
#         eer_list.append(eer)

#     avg_eer = sum(eer_list) / len(eer_list)
#     print(f"{'With' if isomap_applied else 'Without'} Isomap:")
#     print(f"Average Equal Error Rate (EER): {avg_eer * 100:.2f}%")
#     print()
from sklearn.metrics import roc_curve, auc
from scipy.optimize import brentq
from scipy.interpolate import interp1d

def calculate_eer(y_test, y_score):
    fpr, tpr, thresholds = roc_curve(y_test, y_score)
    fnr = 1 - tpr
    eer = brentq(lambda x : 1. - x - interp1d(fpr, fnr)(x), 0., 1.)
    return eer

def run_random_forest(data, user_labels, isomap_applied=False):
    eer_list = []

    for target_user in user_labels:
        user_data = data[data['label'] == target_user]
        impostor_data = data[data['label'] != target_user]

        min_samples = min(user_data.shape[0], impostor_data.shape[0])

        balanced_user_data = user_data.sample(min_samples)
        balanced_impostor_data = impostor_data.sample(min_samples)

        balanced_data = pd.concat([balanced_user_data, balanced_impostor_data]).sample(frac=1).reset_index(drop=True)

        balanced_data['label'] = np.where(balanced_data['label'] == target_user, 0, 1)

        X = balanced_data.drop(['label'], axis=1)
        y = balanced_data['label']

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
        rf_classifier.fit(X_train, y_train)

        y_score = rf_classifier.predict_proba(X_test)[:, 1]

        eer = calculate_eer(y_test, y_score)
        eer_list.append(eer)

        print(f"User: {target_user}, EER: {eer * 100:.2f}%")

    avg_eer = sum(eer_list) / len(eer_list)
    print(f"{'With' if isomap_applied else 'Without'} Isomap:")
    print(f"Average Equal Error Rate (EER): {avg_eer * 100:.2f}%")
    print()



In [14]:
imputer = SimpleImputer(missing_values=np.nan, strategy='mean')
data.iloc[:, :] = imputer.fit_transform(data.replace([np.inf, -np.inf], np.nan))

In [27]:
from sklearn.metrics import pairwise_distances
from sklearn.decomposition import PCA




def apply_pca_per_user(data, user_labels, n_components=10):
    transformed_data_list = []

    for target_user in user_labels:
        user_data = data[data['label'] == target_user]

        pca = PCA(n_components=n_components)
        user_transformed_data = pca.fit_transform(user_data.drop(['label'], axis=1))
        user_transformed_data = pd.DataFrame(user_transformed_data)
        #user_transformed_data['user'] = user_data['user'].values
        user_transformed_data['label'] = user_data['label'].values

        transformed_data_list.append(user_transformed_data)

    return pd.concat(transformed_data_list).reset_index(drop=True)



# Run the Random Forest model without dimensionality reduction
run_random_forest(data, user_labels)

# Apply PCA for dimensionality reduction for each user
data_transformed = apply_pca_per_user(data, user_labels, n_components=10)

# Run the Random Forest model with PCA-transformed data
run_random_forest(data_transformed, user_labels, isomap_applied=True)

User: 1, EER: 100.00%
User: 10, EER: 100.00%
User: 11, EER: 100.00%
User: 12, EER: 100.00%
User: 13, EER: 100.00%
User: 14, EER: 100.00%
User: 15, EER: 100.00%
User: 16, EER: 100.00%
User: 17, EER: 100.00%
User: 18, EER: 0.00%
User: 19, EER: 100.00%
User: 2, EER: 100.00%
User: 3, EER: 100.00%
User: 4, EER: 100.00%
User: 5, EER: 100.00%
User: 6, EER: 100.00%
User: 7, EER: 100.00%
User: 8, EER: 100.00%
User: 9, EER: 100.00%
Without Isomap:
Average Equal Error Rate (EER): 94.74%

User: 1, EER: 100.00%
User: 10, EER: 100.00%
User: 11, EER: 100.00%
User: 12, EER: 100.00%
User: 13, EER: 100.00%
User: 14, EER: 100.00%
User: 15, EER: 100.00%
User: 16, EER: 100.00%
User: 17, EER: 100.00%
User: 18, EER: 0.00%
User: 19, EER: 100.00%
User: 2, EER: 100.00%
User: 3, EER: 100.00%
User: 4, EER: 100.00%
User: 5, EER: 100.00%
User: 6, EER: 100.00%
User: 7, EER: 100.00%
User: 8, EER: 100.00%
User: 9, EER: 100.00%
With Isomap:
Average Equal Error Rate (EER): 94.74%



In [44]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

def run_random_forest(data, user_labels, pca_applied=False):
    all_y_test = []
    all_y_pred = []

    for target_user in user_labels:
        user_data = data[data['label'] == target_user]
        impostor_data = data[data['label'] != target_user]

        min_samples = min(user_data.shape[0], impostor_data.shape[0])

        balanced_user_data = user_data.sample(min_samples)
        balanced_impostor_data = impostor_data.sample(min_samples)

        balanced_data = pd.concat([balanced_user_data, balanced_impostor_data]).sample(frac=1).reset_index(drop=True)

        balanced_data['label'] = np.where(balanced_data['label'] == target_user, 0, 1)

        X = balanced_data.drop(['label'], axis=1)
        y = balanced_data['label']

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
        rf_classifier.fit(X_train, y_train)

        y_pred = rf_classifier.predict(X_test)

        accuracy = accuracy_score(y_test, y_pred)
        print(f"User {target_user} accuracy: {accuracy:.2f}")

        all_y_test.extend(y_test)
        all_y_pred.extend(y_pred)

    print(f"{'With' if pca_applied else 'Without'} PCA:")
    print(classification_report(all_y_test, all_y_pred))
    print()

def apply_pca_to_whole_data(data, n_components=10):
    pca = PCA(n_components=n_components)
    transformed_data = pca.fit_transform(data.drop(['label'], axis=1))
    transformed_data = pd.DataFrame(transformed_data)
    transformed_data['label'] = data['label'].values

    return transformed_data.reset_index(drop=True)


# Run the Random Forest model without dimensionality reduction
run_random_forest(data, user_labels)

# Apply PCA for dimensionality reduction to the entire dataset
data_transformed = apply_pca_to_whole_data(data, n_components=10)

# Run the Random Forest model with PCA-transformed data
run_random_forest(data_transformed, user_labels, pca_applied=True)

User 1 accuracy: 0.92
User 10 accuracy: 0.76
User 11 accuracy: 0.78
User 12 accuracy: 0.76
User 13 accuracy: 0.74
User 14 accuracy: 0.72
User 15 accuracy: 0.95
User 16 accuracy: 0.91
User 17 accuracy: 0.92
User 18 accuracy: 0.50
User 19 accuracy: 0.84
User 2 accuracy: 0.97
User 3 accuracy: 0.87
User 4 accuracy: 0.83
User 5 accuracy: 0.73
User 6 accuracy: 0.90
User 7 accuracy: 0.84
User 8 accuracy: 0.79
User 9 accuracy: 0.71
Without PCA:
              precision    recall  f1-score   support

           0       0.81      0.87      0.84      7671
           1       0.86      0.79      0.82      7625

    accuracy                           0.83     15296
   macro avg       0.83      0.83      0.83     15296
weighted avg       0.83      0.83      0.83     15296


User 1 accuracy: 0.87
User 10 accuracy: 0.63
User 11 accuracy: 0.67
User 12 accuracy: 0.67
User 13 accuracy: 0.65
User 14 accuracy: 0.61
User 15 accuracy: 0.85
User 16 accuracy: 0.78
User 17 accuracy: 0.82
User 18 accuracy: 1.00
Us